# 02 — Cleaning walkthrough (M1 ingest + M2 clean, FR-01..FR-04)
Run the cells in order. Each markdown cell says what output to expect.

In [ ]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
from src.utils import load_config, set_seed
config = load_config()
set_seed(config["seed"])
RAW_DIR = PROJECT_ROOT / "data" / "raw"

## Step 1 — load and validate (FR-01, FR-02)
Expected (real files): traffic ≈ 178,468 rows, weather 10,872, calendar 151. Two date formats parsed into one tz-aware timestamp.

In [ ]:
from src.ingest import load_all
traffic, weather, calendar, quarantine, dq_log = load_all(RAW_DIR, config)
print(traffic.shape, weather.shape, calendar.shape)
print(traffic['timestamp'].dtype, weather['timestamp'].dtype)

## Step 2 — dedup on road_id + date + time, before anything else
Expected: 1,767 duplicates removed → 176,701 unique rows.

In [ ]:
from src.clean import deduplicate_traffic
traffic, duplicate_rows = deduplicate_traffic(traffic, dq_log)
print(len(traffic), len(duplicate_rows))
assert traffic.duplicated(['road_id', 'timestamp']).sum() == 0

## Step 3 — sensor faults → NaN (logged and quarantined)
Expected: 241 negative volumes, 237 speeds > 200, 234 occupancies > 100.

In [ ]:
from src.clean import apply_outlier_rules, flag_statistical_outliers, parse_vehicle_mix
traffic, fault_rows = apply_outlier_rules(traffic, dq_log)
traffic = flag_statistical_outliers(traffic, config, dq_log)
traffic = parse_vehicle_mix(traffic, dq_log)
print(traffic[['traffic_volume', 'avg_speed', 'occupancy']].describe().loc[['min', 'max']])

## Step 4 — reindex to the full 30-min grid (before any lag)
Expected: 25 × 151 × 48 = 181,200 rows; 4,499 windows added.

In [ ]:
from src.clean import reindex_to_full_grid
from src.utils import assign_split
grid = reindex_to_full_grid(traffic, config, dq_log)
grid['split'] = assign_split(grid['timestamp'], config)
print(len(grid), int((~grid['row_observed']).sum()))
print(grid['split'].value_counts())

## Step 5 — repair from vehicle_count, then impute
Short gaps (≤ 2 windows): time interpolation inside the segment. Longer gaps: segment × time-window median from TRAIN dates only. Expected: no NaN left in traffic_volume.

In [ ]:
from src.clean import repair_volume_from_count, impute_series_columns, IMPUTED_COLUMNS
grid = repair_volume_from_count(grid, dq_log)
grid = impute_series_columns(grid, IMPUTED_COLUMNS, config, dq_log)
print(grid['volume_source'].value_counts())
assert grid['traffic_volume'].isna().sum() == 0

## Step 6 — congestion: keep original labels, derive blanks from V/C
V/C = volume ÷ (capacity ÷ 2). Expected agreement with existing labels ≈ 99.9 % — evidence that congestion is a function of same-window volume (the leakage risk).

In [ ]:
from src.clean import fill_congestion
grid['volume_target_observed'] = grid['volume_source'] == 'observed'
grid['accident_observed'] = grid['row_observed'] & grid['accident_count'].notna()
grid = fill_congestion(grid, config, dq_log)
print(dq_log.entries[-3]['note'])
print(grid['congestion_level'].value_counts())

## Step 7 — weather labels and merge
Expected: 4 labels (Clear/Cloudy/Rain/Fog); Overcast counted separately; 181,200 rows after merge.

In [ ]:
from src.clean import harmonise_weather, merge_sources
weather_clean = harmonise_weather(weather, config, dq_log)
print(weather_clean['weather_condition'].value_counts())
clean = merge_sources(grid, weather_clean, calendar, dq_log)
print(clean.shape)

## The whole of M1 + M2 in one call (what the pipeline runs)

In [ ]:
from src.clean import run_cleaning
clean, dq_log = run_cleaning(RAW_DIR, config)
print(dq_log.to_markdown_table()[:2000])